---
# ⚠️ Run this FIRST if `drive.mount` failed

`ValueError: mount failed` is an **authorisation** failure, not a code failure. Isolating the mount into its own cell makes the popup easier to complete.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
p = Path('/content/drive/MyDrive/Component_01')
print('Component_01 on Drive:', p.exists())
if p.exists():
    for f in ['stage6_acr.py', 'cxr_transforms.py', 'training_manifest',
              'checkpoints/stage5/best.pt', 'data/images/cardio_384.tar',
              'data/raw/mimic-cxr-2.0.0-metadata.csv']:
        t = p / f
        mb = (' %.0f MB' % (t.stat().st_size / 1e6)) if t.is_file() else ''
        print(('  OK   ' if t.exists() else '  MISSING '), f, mb)

# STAGE 6 · Acquisition-Conditioned Reliability (ACR)

### ★ This stage is the independent contribution.

Panel feedback on Progress 1 was: *"Other than the existing coding, there is no evidence of any proper independent contribution to the component."*

Stages 1–5 assemble existing work (ConvNeXt, BART, CheXpert, Grad-CAM) — correctly, and with real engineering, but they are assembly. **This stage is a method that did not exist before it was written here.**

---

## The problem

A chest radiograph is not a neutral observation of a patient. It is a measurement made under conditions. Radiologists qualify every finding by those conditions:

> *"Cardiac silhouette appears enlarged, **though assessment is limited by AP technique and low lung volumes**."*

Deployed CXR classifiers do not. They emit `Cardiomegaly 0.94` with **identical confidence** on a perfect PA film and on a rotated, poorly-inspired portable AP.

Measured in **your own data**, across 45,558 films:

| Pathology | AP | PA | Ratio |
|---|---|---|---|
| **Cardiomegaly** | 62.1% | 32.0% | **1.94×** |
| Edema | 32.6% | 7.8% | 4.19× |
| Pleural Effusion | 40.3% | 16.5% | 2.44× |

And the gap survives controlling for co-pathology burden — among patients with **zero** other findings, AP still shows **43.6% vs 21.8%**.

## What ACR does — no retraining, CPU only

1. **AUDIT** — how much apparent performance is projection separation, not anatomy
2. **RECALIBRATE** — per-pathology, conditional on acquisition
3. **SCORE** — attach an *empirical* reliability tier to every prediction
4. **QUALIFY** — emit the radiologist-style hedge in plain English

## ⚠️ The claim we make, and the one we refuse to make

| | |
|---|---|
| ❌ **NOT claimed** | "a non-zero acquisition coefficient proves a shortcut" — it does not. AP patients genuinely *are* sicker. |
| ✅ **Claim 1** | within-projection vs pooled AUROC decomposes how much discrimination survives when the AP/PA cue is removed |
| ✅ **Claim 2** | given the classifier's own probability, acquisition *still* predicts the label ⟹ the classifier is **miscalibrated conditional on acquisition**. Measurable, fixable, worth fixing. |

Every effect is reported with a bootstrap CI **and** a shuffled-acquisition negative control. If the control does not collapse, the effect is an artifact and we say so.

---
# 0 · Config

**Runs locally on CPU by default.** Set `ON_COLAB = True` only if you'd rather spend ~0.5 CU to save ~1 hour.

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ON_COLAB   = True         # True -> mount Drive + L4 GPU   |   False -> local CPU
SMOKE_TEST = True         # True -> 300 images/split (~2 min). ALWAYS run this first.

if ON_COLAB:
    # Mount defensively. drive.mount() raises ValueError('mount failed') if the
    # OAuth popup is blocked, dismissed, or answered with the wrong account --
    # and re-calling it on an ALREADY-mounted drive can fail the same way.
    from google.colab import drive
    if Path('/content/drive/MyDrive').exists():
        print('  Drive already mounted - skipping mount()')
    else:
        try:
            drive.mount('/content/drive')
        except Exception as e:
            raise RuntimeError(
                'Drive mount failed: ' + str(e) + '\n'
                '  1. Re-run this cell and COMPLETE the popup (allow popups for\n'
                '     colab.research.google.com).\n'
                '  2. Pick the Google account that owns MyDrive/Component_01.\n'
                '  3. Grant ALL requested permissions - partial consent fails.\n'
                '  4. Still failing? Runtime > Disconnect and delete runtime,\n'
                '     reconnect, then run the standalone mount cell FIRST.')
    PROJECT  = Path('/content/drive/MyDrive/Component_01')
    IMG_ROOT = Path('/content/cardio_image_384')      # local SSD, NOT Drive
    METADATA = PROJECT / 'data' / 'raw' / 'mimic-cxr-2.0.0-metadata.csv'
    TAR      = PROJECT / 'data' / 'images' / 'cardio_384.tar'
else:
    HERE = Path.cwd()
    PROJECT  = HERE if (HERE / 'training_manifest').exists() else HERE / 'Component_01'
    IMG_ROOT = PROJECT.parent / 'data' / 'output' / 'cardio_image_384'
    METADATA = PROJECT.parent / 'data' / 'raw' / 'mimic-cxr-2.0.0-metadata.csv'
    TAR      = None

MANIFEST = PROJECT / 'training_manifest'
CKPT     = PROJECT / 'checkpoints' / 'stage5' / 'best.pt'
OUT      = PROJECT / 'reports' / 'stage6'; OUT.mkdir(parents=True, exist_ok=True)
CACHE    = OUT / 'cache'; CACHE.mkdir(exist_ok=True)   # on Drive -> survives a restart

sys.path.insert(0, str(PROJECT))

print('  PROJECT ', PROJECT)
print('  IMG_ROOT', IMG_ROOT)
print('  METADATA', METADATA, '->', METADATA.exists())
print('  CKPT    ', CKPT, '->', CKPT.exists())
print('  mode    ', 'COLAB/GPU' if ON_COLAB else 'LOCAL/CPU', '  smoke =', SMOKE_TEST)
if ON_COLAB:
    import subprocess
    print(' ', subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
          '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

---
# 0b · Colab only — stage the images on local SSD

**This is the slowest step of the whole run (~7–11 min) and it dominates the cost.**

The tar is copied off Drive *first*, then extracted locally. Extracting straight from Drive is what failed during Stage 4B — Drive dropped the mount mid-extraction and `tar` died with `CalledProcessError exit 2`.

> Skipped automatically if the images are already staged, so a **Restart runtime** costs you nothing. Only *Disconnect and delete runtime* wipes them.

In [ ]:
if ON_COLAB and not IMG_ROOT.exists():
    import subprocess, shutil
    t0 = time.time()
    local_tar = Path('/content/cardio_384.tar')
    if not local_tar.exists():
        assert TAR.exists(), f'tar not found on Drive: {TAR}'
        gb = TAR.stat().st_size / 1e9
        print('  copying %.1f GB off Drive (avoids the Stage-4B mid-extract drop)...' % gb)
        shutil.copy(TAR, local_tar)
        print('  copied in %.1f min' % ((time.time() - t0) / 60))
    print('  extracting...')
    subprocess.run(['tar', '-xf', str(local_tar), '-C', '/content'], check=True)
    print('  extracted, total %.1f min' % ((time.time() - t0) / 60))
    try:
        local_tar.unlink()          # reclaim ~5 GB of the local disk
    except OSError:
        pass

if ON_COLAB and not IMG_ROOT.exists():
    # tar root differs from the assumed name -- find it rather than fail cryptically
    cand = [p for p in Path('/content').glob('*') if p.is_dir() and (p / 'test').is_dir()]
    assert cand, 'extraction produced no directory containing test/ under /content'
    IMG_ROOT = cand[0]
    print('  IMG_ROOT corrected ->', IMG_ROOT)

print('  images staged at', IMG_ROOT, '->', IMG_ROOT.exists())

---
# 1 · Gate: method self-test

**24 unit tests on the method itself, including two that matter most:** ACR must recover a *known injected bias* with the correct sign, and must measurably improve calibration on it. If this cell fails, nothing below is trustworthy — stop.

In [ ]:
import stage6_acr as acr
importlib = __import__('importlib'); importlib.reload(acr)

np_, nf = acr._selftest()
assert nf == 0, f'{nf} self-tests FAILED — fix stage6_acr.py before continuing'
print(f'\n  ✅ METHOD VALIDATED — {np_} tests passed')

---
# 2 · Integrity gates on the inputs

Every earlier stage in this project was broken by something silent — vanishing image paths, ambiguous label columns, a partial join. Assert first.

In [ ]:
import pandas as pd, numpy as np

PATH = acr.PATHOLOGIES
man = {s: pd.read_csv(MANIFEST / f'manifest_{s}.csv', low_memory=False) for s in ('val','test')}
meta = pd.read_csv(METADATA, low_memory=False)

G = []
def gate(n, ok, v=''):
    G.append(ok); print(f'  {"PASS" if ok else "FAIL"}  {n:<52}{v}')

gate('val + test manifests loaded', all(len(d) for d in man.values()),
     f"val={len(man['val'])} test={len(man['test'])}")
gate('all 8 pathology columns present',
     all(p in man['test'].columns for p in PATH))
gate('labels are strictly 0/1 (no NaN, no -1)',
     all(set(np.unique(d[p])) <= {0,1} for d in man.values() for p in PATH))
gate('no patient overlap val/test',
     len(set(man['val'].subject_id) & set(man['test'].subject_id)) == 0)

for s, d in man.items():
    miss = sum(not (IMG_ROOT / p).exists() for p in d.image_path.head(500))
    gate(f'{s}: image paths resolve (first 500)', miss == 0, f'{500-miss}/500')

gate('metadata has every required column',
     all(c in meta.columns for c in ['dicom_id','ViewPosition',
         'PerformedProcedureStepDescription','StudyTime']))
gate('checkpoint present', CKPT.exists(),
     f'{CKPT.stat().st_size/1e6:.0f} MB' if CKPT.exists() else '← DOWNLOAD IT (see §3)')

assert all(G[:-1]), 'input integrity gate failed'
print('\n  ✅ inputs verified' if all(G) else '\n  ⚠️ checkpoint missing — see the next cell')

---
# 3 · ⚠️ The one prerequisite

Stage 6 needs the **Stage 5 classifier**, which lives on your Drive, not your laptop.

**Download once (~350 MB), then Stage 6 never needs the internet again:**

```
MyDrive/Component_01/checkpoints/stage5/best.pt
        → Component_01/checkpoints/stage5/best.pt
MyDrive/Component_01/checkpoints/stage5/thresholds.json
        → Component_01/checkpoints/stage5/thresholds.json
```

Everything else — images (11 GB), manifests, DICOM metadata — is **already on your machine**. Verified above.

---
# 4 · Acquisition covariates

Two sources, and the code names them honestly:

| | source | exact or proxy |
|---|---|---|
| `is_AP`, `is_portable`, `off_hours` | DICOM headers | **exact** |
| `insp_lungfrac`, `pen_*`, `rot_lr_asym`, `blur_lapvar` | pixels, classical CV | **proxy** |

No segmentation masks. No CheXmask download. No network. ~1 ms/image.

> Image features are computed on the **raw uint8 PNG**, never the z-scored tensor — `PerImageZScore` deliberately destroys absolute intensity, which is exactly what the penetration proxies measure.

In [ ]:
N_SMOKE = 300
acq = {}
for s, d in man.items():
    d = d.head(N_SMOKE).copy() if SMOKE_TEST else d.copy()
    a = acr.metadata_acquisition(d, meta)          # raises on a partial join
    f = CACHE / f'imgfeat_{s}{"_smoke" if SMOKE_TEST else ""}.parquet'
    if f.exists():
        imf = pd.read_parquet(f); print(f'  {s}: image features loaded from cache')
    else:
        t0 = time.time()
        imf = acr.image_acquisition_batch(
            [str(IMG_ROOT / p) for p in d.image_path],
            progress=lambda i, n: print(f'\r  {s}: {i}/{n}  '
                f'({(time.time()-t0)/i*(n-i):.0f}s left)', end=''))
        imf.to_parquet(f)
        print(f'\r  {s}: {len(imf)} images in {time.time()-t0:.0f}s' + ' '*20)
    acq[s] = pd.concat([a.reset_index(drop=True), imf.reset_index(drop=True)], axis=1)
    acq[s]['stratum'] = acr.stratum_id(acq[s])
    man[s] = d.reset_index(drop=True)

print('\n  stratum distribution (test):')
print(acq['test'].stratum.value_counts().to_string())
print('\n  acquisition feature summary (test):')
print(acq['test'][acr.ACQ_FEATURES].describe().T[['mean','std','min','max']].round(4).to_string())

---
# 5 · Classifier inference → cached probabilities

**This is the only slow step, and it happens once.** Everything after it is tabular maths that runs in seconds.

| where | val+test = 9,196 images | cost |
|---|---|---|
| **your laptop (measured 2.6 img/s)** | **~60 min** | **free** |
| Colab L4 | ~1 min compute, ~20 min session | ~0.5 CU |

Writes a chunked cache — if the laptop sleeps or the kernel dies, re-running **resumes** instead of restarting.

In [ ]:
import torch, torchvision
from PIL import Image
from cxr_transforms import build_transform

DEV = 'cuda' if (ON_COLAB and torch.cuda.is_available()) else 'cpu'
torch.set_num_threads(os.cpu_count() or 4)
TF  = build_transform('test')     # MUST match Stage 5 eval exactly
# L4 has 24 GB; ConvNeXt-Base @384 inference at bs=64 uses well under half.
BATCH = 64 if DEV == 'cuda' else 8
print('  device', DEV, '| batch', BATCH)

class CXRClassifier(torch.nn.Module):
    """EXACT replica of Stage 5 (Stage5_Classifier_Training.ipynb, cell 15).

    Verified against best.pt: classifier.0=LayerNorm(1024),
    classifier.2=Linear(1024,512), classifier.5=Linear(512,8).

    DO NOT substitute torchvision's ConvNeXt with a swapped classifier.
    torchvision applies `classifier` to the 4-D (B,1024,1,1) avgpool output;
    Stage 5 flattens FIRST. nn.LayerNorm(1024) on a 4-D tensor normalises the
    LAST dim (size 1) instead of the channel dim -- it does not raise, it
    silently returns wrong numbers that still look like probabilities.
    """
    def __init__(self, n=8, p_drop=0.3):
        super().__init__()
        base = torchvision.models.convnext_base(weights=None)
        self.features = base.features
        self.avgpool  = base.avgpool
        d = base.classifier[2].in_features                     # 1024
        self.classifier = torch.nn.Sequential(
            torch.nn.LayerNorm(d), torch.nn.Dropout(p_drop),
            torch.nn.Linear(d, 512), torch.nn.GELU(),
            torch.nn.Dropout(p_drop * 0.66), torch.nn.Linear(512, n))

    def forward(self, x):
        return self.classifier(self.avgpool(self.features(x)).flatten(1))

def load_stage5(path):
    """Loads STRICTLY. A partially-loaded backbone produces plausible-looking
    garbage, so any key mismatch is a hard stop."""
    ck = torch.load(path, map_location='cpu', weights_only=False)
    got = ck.get('pathologies')
    if got is not None and list(got) != list(PATH):
        raise RuntimeError('pathology ORDER differs -> all 8 columns would be '
                           'permuted silently. ckpt=' + str(got) + ' ours=' + str(PATH))
    m = CXRClassifier(len(PATH))
    sd = {(k[7:] if k.startswith('module.') else k): v for k, v in ck['model'].items()}
    m.load_state_dict(sd, strict=True)            # raises on ANY mismatch
    if ck.get('ema'):                             # Stage 5 reported EMA weights
        msd = m.state_dict()
        miss = [k for k in ck['ema'] if k not in msd]
        if miss:
            raise RuntimeError('EMA keys absent from model: ' + str(miss[:5]))
        for k, v in ck['ema'].items():
            msd[k].copy_(v)
        print('  EMA weights applied (matches Stage 5 reported metrics)')
    print('  loaded epoch', ck.get('epoch'), '| val best_metric',
          round(float(ck.get('best_metric', float('nan'))), 4))
    m = m.eval().to(DEV)
    if DEV == 'cuda':
        m = m.to(memory_format=torch.channels_last)
    return m

@torch.no_grad()
def infer(paths, model, bs=None, tag='', chunk=500, cache_key=''):
    bs = bs or BATCH
    cf = CACHE / ('probs_' + cache_key + '.npy')
    done = np.load(cf) if cf.exists() else np.zeros((0, len(PATH)), np.float32)
    if len(done) >= len(paths):
        print('  ' + tag + ': cached (' + str(len(done)) + ')')
        return done[:len(paths)]
    out, t0 = list(done), time.time()
    for i in range(len(done), len(paths), bs):
        x = torch.stack([TF(Image.open(p)) for p in paths[i:i+bs]]).to(DEV)
        if DEV == 'cuda':
            x = x.to(memory_format=torch.channels_last)
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=(DEV == 'cuda')):
            logits = model(x)
        out += list(torch.sigmoid(logits.float()).cpu().numpy())
        if len(out) % chunk < bs or len(out) >= len(paths):
            np.save(cf, np.array(out, np.float32))
        el = time.time() - t0
        rate = (len(out) - len(done)) / max(el, 1e-9)
        left = (len(paths) - len(out)) / max(rate, 1e-9) / 60
        print('\r  %s: %d/%d  %.1f img/s  %.1f min left'
              % (tag, len(out), len(paths), rate, left), end='')
    np.save(cf, np.array(out, np.float32))
    print()
    return np.array(out, np.float32)

model = load_stage5(CKPT)
probs = {}
for s in ('val','test'):
    key = s + ('_smoke' if SMOKE_TEST else '')
    p = infer([str(IMG_ROOT / q) for q in man[s].image_path], model,
              tag=s, cache_key=key)
    probs[s] = pd.DataFrame(p, columns=PATH)

labels = {s: man[s][PATH].astype(int).reset_index(drop=True) for s in ('val','test')}
_mA = np.nanmean([acr.auroc(labels['test'][k], probs['test'][k]) for k in PATH])
print()
print('  sanity - mean AUROC on test: %.4f' % _mA,
      '   (Stage 5: 0.8554 test / 0.8483 val)')

---
# 6 · EXPERIMENT A — the shortcut audit

**How well does acquisition *alone* predict each pathology?** No pixels of anatomy — just "what kind of film is this".

This is the AUROC a "model" achieves by **detecting the camera rather than the patient**. Cross-validated, so it is an honest out-of-sample number and not a fit statistic.

In [ ]:
print('='*82); print('  ACQUISITION-ONLY AUROC  (no anatomy — the free shortcut)'); print('='*82)
print(f"  {'pathology':<20}{'shortcut AUROC':>16}{'95% CI':>22}{'classifier':>13}{'ratio':>8}")
print('  '+'-'*79)
A = {}
for k in PATH:
    r = acr.shortcut_signal(labels['test'][k], acq['test'])
    c = acr.auroc(labels['test'][k], probs['test'][k])
    A[k] = dict(shortcut=r['auroc'], lo=r['lo'], hi=r['hi'], classifier=c,
                ratio=float((r['auroc']-0.5)/max(c-0.5, 1e-9)))
    print(f"  {k:<20}{r['auroc']:>16.4f}   [{r['lo']:.4f}, {r['hi']:.4f}]{c:>13.4f}{A[k]['ratio']:>8.2f}")
print('  '+'-'*79)
print('\n  ratio = (shortcut-0.5)/(classifier-0.5): the fraction of the classifier\'s')
print('  discriminative margin that acquisition alone already accounts for.')

---
# 7 · EXPERIMENT B — within-projection decomposition ★

**This is Claim 1, and the core scientific result of Stage 6.**

Pooled AUROC lets the model separate AP from PA and score points for it. Within a single projection that cue is gone. If pooled ≫ within, part of the reported performance was never anatomy.

> Standard evaluation **cannot see this** — train and test share the same projection mix, so the shortcut is rewarded identically in both.

In [ ]:
print('='*90); print('  POOLED vs WITHIN-PROJECTION AUROC'); print('='*90)
print(f"  {'pathology':<20}{'pooled':>9}{'AP only':>10}{'PA only':>10}{'within-wtd':>12}{'gap':>9}")
print('  '+'-'*87)
B = {}
for k in PATH:
    w = acr.within_stratum_auroc(labels['test'][k], probs['test'][k],
                                 np.where(acq['test'].is_AP > .5, 'AP', 'PA'))
    ap = w['strata'].get('AP', {}).get('auroc', np.nan)
    pa = w['strata'].get('PA', {}).get('auroc', np.nan)
    B[k] = w
    flag = '  ←' if w['shortcut_gap'] > 0.02 else ''
    print(f"  {k:<20}{w['pooled']['auroc']:>9.4f}{ap:>10.4f}{pa:>10.4f}"
          f"{w['within_weighted']:>12.4f}{w['shortcut_gap']:>+9.4f}{flag}")
print('  '+'-'*87)
gaps = [B[k]['shortcut_gap'] for k in PATH if np.isfinite(B[k]['shortcut_gap'])]
print(f'\n  mean shortcut gap: {np.mean(gaps):+.4f}   max: {np.max(gaps):+.4f}')
print('  gap > 0 => pooled AUROC overstates anatomical discrimination.')

---
# 8 · EXPERIMENT C — fit ACR, and prove it with a negative control ★

**Claim 2.** Fitted on **validation only** — fitting on train would be invalid, because the classifier saw those images and its probabilities there are overconfident in a way that does not transfer.

The negative control refits on **row-shuffled acquisition**. If the gain survives shuffling, it was free parameters rather than signal, and we report that instead.

In [ ]:
M = acr.ACRModel(PATH, acr.ACQ_FEATURES).fit(probs['val'], labels['val'], acq['val'])
adj = M.apply(probs['test'], acq['test'])

print('='*94); print('  ACR — TEST SET, before vs after'); print('='*94)
print(f"  {'pathology':<20}{'AUROC':>8}{'→':>3}{'AUROC*':>8}{'ΔAUROC [95% CI]':>24}{'ECE':>8}{'→':>3}{'ECE*':>8}")
print('  '+'-'*91)
C = {}
for k in PATH:
    y = labels['test'][k].to_numpy()
    a0, a1 = probs['test'][k].to_numpy(), adj[k].to_numpy()
    d, lo, hi = acr.delta_ci(y, a0, a1)
    e0, e1 = acr.ece(y, a0), acr.ece(y, a1)
    C[k] = dict(auroc=acr.auroc(y,a0), auroc_acr=acr.auroc(y,a1), d=d, lo=lo, hi=hi,
                ece=e0, ece_acr=e1, acq_weight=M.acquisition_weight(k))
    sig = '✅' if lo > 0 else ('❌' if hi < 0 else '·')
    print(f"  {k:<20}{C[k]['auroc']:>8.4f}{'→':>3}{C[k]['auroc_acr']:>8.4f}"
          f"{d:>+10.4f} [{lo:+.4f},{hi:+.4f}] {sig}{e0:>7.4f}{'→':>3}{e1:>8.4f}")
print('  '+'-'*91)
print(f"  {'MEAN':<20}{np.mean([C[k]['auroc'] for k in PATH]):>8.4f}{'→':>3}"
      f"{np.mean([C[k]['auroc_acr'] for k in PATH]):>8.4f}"
      f"{np.mean([C[k]['d'] for k in PATH]):>+10.4f}{'':>18}"
      f"{np.mean([C[k]['ece'] for k in PATH]):>7.4f}{'→':>3}{np.mean([C[k]['ece_acr'] for k in PATH]):>8.4f}")

print('\n  NEGATIVE CONTROL — acquisition rows shuffled:')
nc = acr.negative_control(probs['val'], labels['val'], acq['val'],
                          probs['test'], labels['test'], acq['test'])
real = np.mean([C[k]['auroc_acr'] - C[k]['auroc'] for k in PATH])
fake = np.mean([nc[k]['shuffled'] - nc[k]['base'] for k in PATH])
print(f'    real ΔAUROC {real:+.4f}   shuffled ΔAUROC {fake:+.4f}')
print('    ' + ('✅ control collapses — the gain is acquisition signal'
                if abs(fake) < abs(real)/2 or real <= 0 else
                '⚠️ control did NOT collapse — report the gain as unproven'))

---
# 9 · Subgroup fairness — does ACR close the AP/PA gap?

The practical payoff. A classifier whose probabilities mean different things on AP and PA is unfair across acquisition subgroups. ACR should shrink that.

In [ ]:
print('='*88); print('  AP/PA SUBGROUP CALIBRATION GAP  |ECE_AP − ECE_PA|'); print('='*88)
print(f"  {'pathology':<20}{'before':>12}{'after':>12}{'change':>12}")
print('  '+'-'*85)
ap_m = (acq['test'].is_AP > .5).to_numpy()
D = {}
for k in PATH:
    y = labels['test'][k].to_numpy()
    g0 = abs(acr.ece(y[ap_m], probs['test'][k].to_numpy()[ap_m]) -
             acr.ece(y[~ap_m], probs['test'][k].to_numpy()[~ap_m]))
    g1 = abs(acr.ece(y[ap_m], adj[k].to_numpy()[ap_m]) -
             acr.ece(y[~ap_m], adj[k].to_numpy()[~ap_m]))
    D[k] = dict(before=g0, after=g1)
    print(f"  {k:<20}{g0:>12.4f}{g1:>12.4f}{g1-g0:>+12.4f}")
print('  '+'-'*85)
b, a_ = np.mean([D[k]['before'] for k in PATH]), np.mean([D[k]['after'] for k in PATH])
print(f"  {'MEAN':<20}{b:>12.4f}{a_:>12.4f}{a_-b:>+12.4f}")
print(f"\n  {'✅ subgroup gap reduced' if a_ < b else '⚠️ subgroup gap NOT reduced — report honestly'}"
      f"  ({(1-a_/max(b,1e-9))*100:+.1f}%)")

---
# 10 · The reliability score — and the test that could falsify it

Reliability is **not invented**. It is the classifier's own measured discrimination in that acquisition stratum, fitted on validation.

**The falsification test:** drop the least-reliable films and re-measure AUROC. A score that means something produces a **rising** curve. A flat curve means the score is noise — and we would report that.

In [ ]:
RT = acr.ReliabilityTable.fit(labels['val'], probs['val'], acq['val'].stratum, PATH,
                              min_n=20 if SMOKE_TEST else 100)
print('='*80); print('  VALIDATION AUROC BY ACQUISITION STRATUM'); print('='*80)
strata = sorted(acq['val'].stratum.unique())
print(f"  {'pathology':<20}" + ''.join(f'{s:>15}' for s in strata) + f"{'pooled':>10}")
print('  '+'-'*77)
for k in PATH:
    print(f'  {k:<20}' + ''.join(f'{RT.score(k,s):>15.4f}' for s in strata)
          + f'{RT.overall[k]:>10.4f}')

print('\n' + '='*80); print('  FALSIFICATION TEST — selective prediction (Cardiomegaly)'); print('='*80)
rel = np.array([RT.score('Cardiomegaly', s) for s in acq['test'].stratum])
sc = acr.selective_curve(labels['test']['Cardiomegaly'], adj['Cardiomegaly'], rel)
print(sc.round(4).to_string(index=False))
rise = sc.auroc.iloc[0] - sc.auroc.iloc[-1]
print(f'\n  AUROC at 10% most-reliable minus at 100%: {rise:+.4f}')
print('  ' + ('✅ reliability predicts error — the score is meaningful' if rise > 0.01
              else '⚠️ curve is flat — the score does NOT predict error. Report this.'))

---
# 11 · The user-visible output

What Stage 8 will actually render. The qualification is **templated, not generated** — a language model here could hallucinate a limitation that was never measured, which is exactly the failure this component exists to remove.

In [ ]:
THR = {'insp_lungfrac': float(acq['val'].insp_lungfrac.quantile(.25)),
       'rot_lr_asym':   float(acq['val'].rot_lr_asym.quantile(.75)),
       'pen_mean':      float(acq['val'].pen_mean.quantile(.75)),
       'blur_lapvar':   float(acq['val'].blur_lapvar.quantile(.25))}
print('  proxy thresholds (val quantiles):', {k: round(v,4) for k,v in THR.items()}, '\n')

show = acq['test'][acq['test'].is_AP > .5].head(3).index.tolist() + \
       acq['test'][acq['test'].is_AP <= .5].head(2).index.tolist()
for i in show:
    row, st = acq['test'].loc[i], acq['test'].stratum.iloc[i]
    print('='*78)
    print(f"  film {i} · {st} · hour {row.study_hour:.0f}")
    for k in ('Cardiomegaly','Edema','Pleural_Effusion'):
        raw, a1 = probs['test'][k].iloc[i], adj[k].iloc[i]
        tier = RT.tier(k, st)
        mark = {'HIGH':'✅','MODERATE':'⚠️','LOW':'❌','UNKNOWN':'·'}[tier]
        print(f"    {mark} {k:<18} {raw:.3f} → {a1:.3f}   reliability {tier}"
              f"   (truth {int(labels['test'][k].iloc[i])})")
        q = acr.qualification(row, tier, k, THR)
        if q: print(f"         ↳ {q}")

---
# 12 · Save

In [ ]:
from datetime import datetime
res = dict(stage=6, timestamp=datetime.now().isoformat(), smoke_test=SMOKE_TEST,
           n_val=len(man['val']), n_test=len(man['test']),
           shortcut_audit=A, within_projection={k: {kk: vv for kk, vv in B[k].items()} for k in PATH},
           acr=C, subgroup_gap=D, acr_model=M.to_dict(),
           reliability_table=RT.table, reliability_overall=RT.overall,
           proxy_thresholds=THR,
           negative_control=dict(real_delta=float(real), shuffled_delta=float(fake)),
           selective_curve=sc.to_dict('records'))
suffix = '_smoke' if SMOKE_TEST else ''
(OUT / f'stage6_results{suffix}.json').write_text(
    json.dumps(res, indent=2, default=float), encoding='utf-8')
adj.to_csv(OUT / f'adjusted_probs_test{suffix}.csv', index=False)
acq['test'].to_csv(OUT / f'acquisition_test{suffix}.csv', index=False)
print(f"  ✅ {OUT/f'stage6_results{suffix}.json'}")
print(f"  ✅ {OUT/f'adjusted_probs_test{suffix}.csv'}   ← Stage 7 consumes this")
print(f"  ✅ {OUT/f'acquisition_test{suffix}.csv'}")
if SMOKE_TEST:
    print('\n  ⚠️ SMOKE TEST ONLY. Set SMOKE_TEST = False and re-run for real numbers.')

---
# What you can defend to the panel

| Claim | Evidence in this notebook |
|---|---|
| A projection shortcut exists and standard evaluation hides it | §6 acquisition-only AUROC, §7 pooled-vs-within decomposition |
| The classifier is miscalibrated conditional on acquisition | §8 ECE before/after, with bootstrap CIs |
| The correction is signal, not free parameters | §8 shuffled-acquisition negative control |
| It improves subgroup fairness | §9 AP/PA calibration gap |
| The reliability score is meaningful, not decorative | §10 selective-prediction curve (falsifiable) |
| The method is ours | `stage6_acr.py` — standalone, 24 unit tests, recovers a known injected bias |

**What you must NOT claim:** that a non-zero acquisition coefficient proves shortcut learning. It does not. Say "miscalibrated conditional on acquisition" — it is what was measured, and it is defensible.